In [ ]:
import numpy as np
from IPython.display import display, Image

# --- 1. NETWORK & RADIO PARAMETERS ---
NUM_SENSORS = 50        # Total sensor nodes
FIELD_SIZE = 100        # 100x100 meter simulation area
SINK_POS = np.array([50, 50])  # Base station / Sink location at center

# Radio Model Parameters (Standard LEACH-style values)
E_INIT = 0.5           # Initial energy of each sensor node (Joules)
E_ELEC = 50e-9         # Energy dissipation to run transmitter/receiver electronics (nJ/bit)
E_MP = 0.0013e-12      # Multi-path amplifier energy dissipation (pJ/bit/m^4) for d > d0
E_FS = 10e-12          # Free-space amplifier energy dissipation (pJ/bit/m^2) for d <= d0
D0 = np.sqrt(E_FS / E_MP) # Threshold distance (approx 87-88m)
E_DA = 5e-9            # Data aggregation energy (nJ/bit/signal)
PACKET_SIZE = 4000     # Packet size in bits (4000 bits = 500 bytes)

# GA Configuration
POP_SIZE = 40
GENERATIONS = 100
MUTATION_RATE = 0.05
TARGET_CH_COUNT = 5

# Generate static random positions and initialize energy levels
np.random.seed(42)
SENSOR_POSITIONS = np.random.uniform(0, FIELD_SIZE, size=(NUM_SENSORS, 2))
NODE_ENERGY = np.full(NUM_SENSORS, E_INIT)  # Current remaining energy array

# --- 2. RADIO ENERGY CALCULATION ---
def calculate_tx_energy(distance, bits=PACKET_SIZE):
    """Calculates transmission energy based on distance and standard radio models."""
    if distance < D0:
        return bits * (E_ELEC + E_FS * (distance ** 2))
    else:
        return bits * (E_ELEC + E_MP * (distance ** 4))

def calculate_rx_energy(bits=PACKET_SIZE):
    """Calculates energy expended while receiving data."""
    return bits * E_ELEC

# --- 3. ENERGY-BASED FITNESS FUNCTION ---
def calculate_fitness(chromosome):
    """
    Evaluates fitness based on realistic total energy consumed in 1 transmission round.
    Lower fitness score is better. Dead nodes cannot become Cluster Heads.
    """
    ch_indices = np.where(chromosome == 1)[0]

    # Penalize if no Cluster Heads are chosen, or if a chosen CH is dead
    if len(ch_indices) == 0 or np.any(NODE_ENERGY[ch_indices] <= 0):
        return float('inf')

    total_round_energy = 0.0

    # Step 1: Account for regular nodes transmitting data to their nearest CH
    for i in range(NUM_SENSORS):
        if NODE_ENERGY[i] <= 0:
            continue  # Ignore dead nodes completely

        if chromosome[i] == 1:
            # It's a CH: It consumes energy to aggregate data and send to the central Sink
            dist_to_sink = np.linalg.norm(SENSOR_POSITIONS[i] - SINK_POS)
            tx_to_sink_energy = calculate_tx_energy(dist_to_sink)

            # Aggregate data penalty for every expected child node assigned to it
            # (Roughly approximated by network density distribution)
            aggregation_energy = (NUM_SENSORS / len(ch_indices)) * E_DA * PACKET_SIZE

            total_round_energy += (tx_to_sink_energy + aggregation_energy)
        else:
            # Regular node: find nearest active CH and calculate transmission cost
            distances = [np.linalg.norm(SENSOR_POSITIONS[i] - SENSOR_POSITIONS[ch]) for ch in ch_indices]
            min_dist = min(distances)
            nearest_ch = ch_indices[np.argmin(distances)]

            # Sender transmitter cost + CH receiver cost
            total_round_energy += calculate_tx_energy(min_dist)
            total_round_energy += calculate_rx_energy()

    # Step 2: Multi-objective constraints (Encourage target CH distribution)
    ch_penalty = abs(len(ch_indices) - TARGET_CH_COUNT) * 0.05  # Normalized scaling factor

    return total_round_energy + ch_penalty

# --- 4. GENETIC OPERATIONS ---
def initialize_population():
    population = []
    # Identify alive nodes
    alive_indices = np.where(NODE_ENERGY > 0)[0]

    for _ in range(POP_SIZE):
        chromosome = np.zeros(NUM_SENSORS, dtype=int)
        if len(alive_indices) >= TARGET_CH_COUNT:
            ch_select = np.random.choice(alive_indices, TARGET_CH_COUNT, replace=False)
            chromosome[ch_select] = 1
        population.append(chromosome)
    return np.array(population)

def selection(population, fitness_scores):
    pool = np.random.choice(len(population), size=3, replace=False)
    return population[pool[np.argmin(fitness_scores[pool])]].copy()

def crossover(p1, p2):
    point = np.random.randint(1, NUM_SENSORS - 1)
    return np.concatenate((p1[:point], p2[point:])), np.concatenate((p2[:point], p1[point:]))

def mutate(chromosome):
    for i in range(NUM_SENSORS):
        if np.random.rand() < MUTATION_RATE:
            # Only flip if node is currently alive
            if NODE_ENERGY[i] > 0:
                chromosome[i] = 1 - chromosome[i]
            else:
                chromosome[i] = 0 # Dead nodes stay 0
    return chromosome

# --- 5. EXECUTION PIPELINE ---
def optimize_wsn_round():
    population = initialize_population()
    best_chromosome = None
    best_fitness = float('inf')

    for generation in range(GENERATIONS):
        fitness_scores = np.array([calculate_fitness(ind) for ind in population])

        min_idx = np.argmin(fitness_scores)
        if fitness_scores[min_idx] < best_fitness:
            best_fitness = fitness_scores[min_idx]
            best_chromosome = population[min_idx].copy()

        new_pop = [best_chromosome.copy()]
        while len(new_pop) < POP_SIZE:
            p1 = selection(population, fitness_scores)
            p2 = selection(population, fitness_scores)
            c1, c2 = crossover(p1, p2)
            new_pop.extend([mutate(c1), mutate(c2)])

        population = np.array(new_pop[:POP_SIZE])

    return best_chromosome, best_fitness

if __name__ == "__main__":
    print("--- Running GA Optimization for Energy-Aware WSN ---")
    best_layout, expected_energy_cost = optimize_wsn_round()

    elected_chs = np.where(best_layout == 1)[0]
    print(f"Optimal Cluster Heads Elected: {elected_chs}")
    print(f"Total Projected Energy Expended per Round: {expected_energy_cost:.6f} Joules")
    print(f"Average Initial Energy per Node Remaining: {np.mean(NODE_ENERGY):.4f} J")


display(Image(filename="C:\\Downloads\\WhatsApp Image 2026-10-16 at 12.31.51 AM.jpeg"))
display(Image(filename="C:\\Downloads\\WhatsApp Image 2026-10-16 at 12.31.52 AM.jpeg"))


--- Running GA Optimization for Energy-Aware WSN ---
Optimal Cluster Heads Elected: [18 23 31 33 37]
Total Projected Energy Expended per Round: 0.020990 Joules
Average Initial Energy per Node Remaining: 0.5000 J


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Downloads\\WhatsApp Image 2026-10-16 at 12.31.51 AM.jpeg'

In [ ]:
import numpy as np

# --- 1. NETWORK & GA PARAMETERS ---
NUM_SENSORS = 50       # Total sensor nodes in the field
FIELD_SIZE = 100       # 100x100 meter simulation area
NUM_CH = 5             # Target number of Cluster Heads to elect
SINK_POS = np.array([50, 50])  # Base station / Sink location at center

POP_SIZE = 40          # Chromosomes per generation
GENERATIONS = 100      # Total iterations
MUTATION_RATE = 0.05   # Probability of mutation per gene

# Generate fixed random 2D positions for the sensors
np.random.seed(42)
SENSOR_POSITIONS = np.random.uniform(0, FIELD_SIZE, size=(NUM_SENSORS, 2))

# --- 2. FITNESS FUNCTION ---
def calculate_fitness(chromosome):
    """
    Evaluates a chromosome. Lower fitness is better (minimizing distance/energy cost).
    Returns infinity if the chromosome is invalid (e.g., zero CHs elected).
    """
    # Identify which nodes are selected as Cluster Heads (1s)
    ch_indices = np.where(chromosome == 1)[0]

    if len(ch_indices) == 0:
        return float('inf') # Penalty for invalid network topology

    total_distance_cost = 0

    for i in range(NUM_SENSORS):
        if chromosome[i] == 1:
            # Distance from Cluster Head to the central Sink
            total_distance_cost += np.linalg.norm(SENSOR_POSITIONS[i] - SINK_POS)
        else:
            # Distance from Regular Sensor to its nearest designated Cluster Head
            distances_to_chs = [np.linalg.norm(SENSOR_POSITIONS[i] - SENSOR_POSITIONS[ch]) for ch in ch_indices]
            total_distance_cost += min(distances_to_chs)

    # Multi-objective penalty: discourage deviating wildly from target number of CHs
    ch_penalty = abs(len(ch_indices) - NUM_CH) * FIELD_SIZE

    return total_distance_cost + ch_penalty

# --- 3. GENETIC OPERATORS ---
def initialize_population():
    """Creates a random binary initial population."""
    population = []
    for _ in range(POP_SIZE):
        # Initialize chromosomes with roughly the right proportion of CHs
        chromosome = np.zeros(NUM_SENSORS, dtype=int)
        ch_indices = np.random.choice(NUM_SENSORS, NUM_CH, replace=False)
        chromosome[ch_indices] = 1
        population.append(chromosome)
    return np.array(population)

def selection(population, fitness_scores):
    """Selects a parent using Tournament Selection."""
    pool_indices = np.random.choice(len(population), size=3, replace=False)
    best_index = pool_indices[np.argmin(fitness_scores[pool_indices])]
    return population[best_index].copy()

def crossover(parent1, parent2):
    """Performs single-point crossover."""
    point = np.random.randint(1, NUM_SENSORS - 1)
    child1 = np.concatenate((parent1[:point], parent2[point:]))
    child2 = np.concatenate((parent2[:point], parent1[point:]))
    return child1, child2

def mutate(chromosome):
    """Applies bit-flip mutation based on the mutation rate."""
    for i in range(NUM_SENSORS):
        if np.random.rand() < MUTATION_RATE:
            chromosome[i] = 1 - chromosome[i] # Flip bit (0 -> 1 or 1 -> 0)
    return chromosome

# --- 4. MAIN GENETIC ALGORITHM LOOP ---
def run_wsn_optimization():
    population = initialize_population()

    best_chromosome = None
    best_fitness = float('inf')

    print(f"Starting GA Optimization over {GENERATIONS} generations...")

    for generation in range(GENERATIONS):
        # Evaluate fitness for the current population
        fitness_scores = np.array([calculate_fitness(ind) for ind in population])

        # Track elitism / best solution found so far
        min_fit_idx = np.argmin(fitness_scores)
        if fitness_scores[min_fit_idx] < best_fitness:
            best_fitness = fitness_scores[min_fit_idx]
            best_chromosome = population[min_fit_idx].copy()

        # Generate new population
        new_population = []
        # Keep the absolute best individual (Elitism)
        new_population.append(best_chromosome.copy())

        while len(new_population) < POP_SIZE:
            parent1 = selection(population, fitness_scores)
            parent2 = selection(population, fitness_scores)

            child1, child2 = crossover(parent1, parent2)

            new_population.append(mutate(child1))
            if len(new_population) < POP_SIZE:
                new_population.append(mutate(child2))

        population = np.array(new_population)

        if (generation + 1) % 10 == 0 or generation == 0:
            print(f"Generation {generation + 1:03d}: Best Fitness Cost = {best_fitness:.2f}")

    # Final Summary
    final_ch_indices = np.where(best_chromosome == 1)[0]
    print("\n--- OPTIMIZATION COMPLETE ---")
    print(f"Optimized Cost: {best_fitness:.2f}")
    print(f"Elected Cluster Heads ({len(final_ch_indices)} nodes): {final_ch_indices}")

if __name__ == "__main__":
    run_wsn_optimization()
